In [2]:
import pandas as pd
books=pd.read_csv("data\/books_cleaned.csv")

In [14]:
books["categories"].value_counts().reset_index().query("count >10")

,categories,count
0,Fiction,2075
1,Juvenile Fiction,377
2,Biography & Autobiography,302
3,History,203
4,Literary Criticism,121
5,Comics & Graphic Novels,116
6,Philosophy,112
7,Religion,112
8,Drama,81
9,Science,55


In [46]:
category_mapping={
    "Fiction":"Fiction",
    "Juvenile Fiction":"Fiction",
    "Drama":"Fiction",
    "Comics&Graphic Novels":"Fiction",
    "Adventure stories":"Fiction",
    "Detective and mystery stories":"Fiction",
    "Fantasy fiction":"Fiction",
    "English fiction":"Fiction",

    "Biography&Autobiography":"Nonfiction",
    "History":"Nonfiction",
    "Children's stories":"Nonfiction",
    "Juvenile Nonfiction":"Nonfiction",
    "True Crime":"Nonfiction",
    "Education":"Nonfiction",
    "Literary Criticism":"Nonfiction",
    "Literary Collections":"Nonfiction",
    "Poetry":"Fiction",
    "Performing Arts":"Nonfiction",
    "Art":"Nonfiction",
    "Humor":"Nonfiction",
    "Language Arts & Disciplines":"Nonfiction",


    "Philosophy":"Nonfiction",
    "Religion":"Nonfiction",
    "Christian life":"Nonfiction",
    "Science": "Nonfiction",
    "Business & Economics":"Nonfiction",
    "Social Science":"Nonfiction",
    "Psychology":"Nonfiction",
    "Political Science":"Nonfiction",
    "Computers":"Nonfiction",
    "Medical":"Nonfiction",
    "Nature":"Nonfiction",

    "Cooking":"Nonfiction",
    "Travel":"Nonfiction",
    "Body, Mind & Spirit":"Nonfiction",
    "Self-Help":"Nonfiction",
    "Health & Fitness":"Nonfiction",
    "Family & Relationships":"Nonfiction",
    "Music":"Nonfiction",
    "Games":"Nonfiction",
    "Sports & Recreation":"Nonfiction"





}

In [47]:
books["simple_categories"]=books["categories"].map(category_mapping)
books["simple_categories"].value_counts().reset_index()

,simple_categories,count
0,Fiction,2638
1,Nonfiction,1312


In [48]:
from transformers import pipeline

categories=["Fiction","Nonfiction"]
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

Device set to use cpu


In [49]:
sequence = books.loc[books["simple_categories"] =="Fiction","description"].reset_index(drop=True)[1]
sequence

"A new 'Christie for Christmas' -- a full-length novel adapted from her acclaimed play by Charles Osborne Following BLACK COFFEE and THE UNEXPECTED GUEST comes the final Agatha Christie play novelisation, bringing her superb storytelling to a new legion of fans. Clarissa, the wife of a Foreign Office diplomat, is given to daydreaming. 'Supposing I were to come down one morning and find a dead body in the library, what should I do?' she muses. Clarissa has her chance to find out when she discovers a body in the drawing-room of her house in Kent. Desperate to dispose of the body before her husband comes home with an important foreign politician, Clarissa persuades her three house guests to become accessories and accomplices. It seems that the murdered man was not unknown to certain members of the house party (but which ones?), and the search begins for the murderer and the motive, while at the same time trying to persuade a police inspector that there has been no murder at all... SPIDER'

In [50]:
classifier(sequence, categories)

{'sequence': "A new 'Christie for Christmas' -- a full-length novel adapted from her acclaimed play by Charles Osborne Following BLACK COFFEE and THE UNEXPECTED GUEST comes the final Agatha Christie play novelisation, bringing her superb storytelling to a new legion of fans. Clarissa, the wife of a Foreign Office diplomat, is given to daydreaming. 'Supposing I were to come down one morning and find a dead body in the library, what should I do?' she muses. Clarissa has her chance to find out when she discovers a body in the drawing-room of her house in Kent. Desperate to dispose of the body before her husband comes home with an important foreign politician, Clarissa persuades her three house guests to become accessories and accomplices. It seems that the murdered man was not unknown to certain members of the house party (but which ones?), and the search begins for the murderer and the motive, while at the same time trying to persuade a police inspector that there has been no murder at a

In [43]:
import numpy as np
max_index= np.argmax(classifier(sequence,categories)["scores"])
max_label= classifier(sequence,categories)["labels"][max_index]
max_label

'Fiction'

In [51]:
def generate_predictions(sequence,categories):
    predictions= classifier(sequence,categories)
    max_index= np.argmax(predictions["scores"])
    max_label=predictions["labels"][max_index]
    return max_label

In [54]:
from tqdm import tqdm
actual_cats=[]
predicted_cats=[]

for i in tqdm(range(0,50)):
    sequence = books.loc[books["simple_categories"]=="Fiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence,categories)]
    actual_cats +=["Fiction"]
for i in tqdm(range(0,50)):
    sequence = books.loc[books["simple_categories"]=="Nonfiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence,categories)]
    actual_cats +=["Nonfiction"]

100%|██████████| 50/50 [00:44<00:00,  1.14it/s]


In [56]:
predicted_df=pd.DataFrame({"actual_categories":actual_cats,"predicted_categories":predicted_cats})


In [58]:
predicted_df["correct_prediction"]=(
    np.where(predicted_df["actual_categories"]==predicted_df["predicted_categories"],1,0)
)

In [60]:
predicted_df["correct_prediction"].sum()/len(predicted_df)

np.float64(0.79)

In [61]:
# predict categories for missing categories
isbns=[]
predicted_cats=[]

missing_cats=books.loc[books["simple_categories"].isna(),["isbn13","description"]].reset_index(drop=True)

In [62]:
for i in tqdm(range(len(missing_cats))):
    sequence= missing_cats["description"][i]
    predicted_cats +=[generate_predictions(sequence,categories)]
    isbns += [missing_cats["isbn13"][i]]

100%|██████████| 1138/1138 [16:33<00:00,  1.15it/s]


In [63]:
missing_predicted_df= pd.DataFrame({"isbn13":isbns,"predicted_categories":predicted_cats})

In [68]:
# now merge into original dataset
books= pd.merge(books, missing_predicted_df,on="isbn13",how="left")
books["simple_categories"] = np.where(books["simple_categories"].isna(), books["predicted_categories"],books["simple_categories"])
books = books.drop(columns=["predicted_categories"])
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web:A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5083,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...,Fiction
5084,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...,Fiction
5085,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Nonfiction
5086,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that:Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction


In [ ]:
#break down fiction into more categories -> not enough labeled dat
books[books["categories"].str.lower().isin([
    "romance",
    "science fiction",
    "scifi",
    "fantasy",
    "horror",
    "mystery",
    "thriller",
    "comedy",
    "crime",
    "historical"


])]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories
24,9780006513087,0006513085,Gravity,Tess Gerritsen,Science fiction,http://books.google.com/books/content?id=KI66c...,Emma Watson a research physician has been trai...,2004.0,4.04,342.0,8024.0,Gravity,9780006513087 Emma Watson a research physician...,Nonfiction
469,9780099410355,0099410354,Traitor,Matthew Woodring Stover,Science fiction,http://books.google.com/books/content?id=VbICO...,"From the depths of catastrophe, a glimmer of h...",2002.0,4.00,320.0,6765.0,Traitor,"9780099410355 From the depths of catastrophe, ...",Fiction
472,9780099422341,0099422344,Yeats is Dead!,Joseph O'Connor,Comedy,http://books.google.com/books/content?id=DrE3I...,"In aid of Amnesty International, this is a bri...",2002.0,3.39,298.0,34.0,Yeats is Dead!:A Novel by Fifteen Irish Writers,"9780099422341 In aid of Amnesty International,...",Fiction
485,9780099446729,0099446723,Blackwood Farm,Anne Rice,Horror,http://books.google.com/books/content?id=cIn8T...,"Lestat Is Back, Saviour And Demon, Presiding O...",2003.0,3.86,774.0,26145.0,Blackwood Farm,"9780099446729 Lestat Is Back, Saviour And Demo...",Fiction
1069,9780261102422,0261102427,The Silmarillion,John Ronald Reuel Tolkien,Fantasy,http://books.google.com/books/content?id=22ePu...,Tolkien's Silmarillion is the core work of the...,1999.0,3.91,384.0,253.0,The Silmarillion,9780261102422 Tolkien's Silmarillion is the co...,Fiction
1411,9780340837955,0340837950,Stranger in a Strange Land,Robert A. Heinlein,Science fiction,http://books.google.com/books/content?id=ZQhiP...,"Epic, entertaining, Stranger in a Strange Land...",2005.0,3.92,672.0,563.0,Stranger in a Strange Land,"9780340837955 Epic, entertaining, Stranger in ...",Fiction
1415,9780345251220,0345251229,Visions from Nowhere,William Arrow,Science fiction,NaN,"The first novel in the series, ""Return to the ...",1976.0,3.23,183.0,10.0,Visions from Nowhere,"9780345251220 The first novel in the series, ""...",Fiction
2782,9780575075597,0575075597,Replay,Ken Grimwood,Fantasy,http://books.google.com/books/content?id=9vmNP...,At forty-three Jeff Winston is tired of his lo...,2005.0,4.16,272.0,412.0,Replay,9780575075597 At forty-three Jeff Winston is t...,Fiction
2797,9780590254762,0590254766,"The lion, the witch and the wardrobe",Clive Staples Lewis,Fantasy,NaN,Four English school children enter the magic l...,1995.0,4.21,189.0,860.0,"The lion, the witch and the wardrobe",9780590254762 Four English school children ent...,Nonfiction
3211,9780739423851,0739423851,Wizard's Castle,Diana Wynne Jones,Fantasy,http://books.google.com/books/content?id=hB7hA...,Howl's moving castle - Eldest of three sisters...,2002.0,4.44,376.0,439.0,Wizard's Castle,9780739423851 Howl's moving castle - Eldest of...,Fiction


In [71]:
books.to_csv("data\/books_with_categories.csv",index=False)